# ISIC18 EDF Learnable Fusion

Kaggle notebook wrapper for the existing file-wise training code. Attach your project repository as a Kaggle input dataset, upload/copy it into `/kaggle/working/Classification`, or set `GITHUB_REPO_URL` in the setup cell. The notebook uses the Kaggle dataset mounted at `/kaggle/input/isic-2018-classification` and runs `src/main.py` directly.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
PROJECT_DIR = KAGGLE_WORKING / 'Classification'

if (Path.cwd() / 'src' / 'main.py').exists():
    PROJECT_DIR = Path.cwd()

# Optional manual settings. Usually leave these blank.
# CODE_INPUT_DIR example: '/kaggle/input/classification-code/Classification'
# GITHUB_REPO_URL example: 'https://github.com/<user>/<repo>.git'
CODE_INPUT_DIR = os.environ.get('CODE_INPUT_DIR', '').strip()
GITHUB_REPO_URL = os.environ.get('GITHUB_REPO_URL', '').strip() or 'https://github.com/AnushtupGhosh5/Classification'

def copy_project_from_input(src_dir, dst_dir):
    src_dir = Path(src_dir)
    if not (src_dir / 'src' / 'main.py').exists():
        raise FileNotFoundError(f'No src/main.py found under {src_dir}')
    if dst_dir.exists():
        return dst_dir
    shutil.copytree(src_dir, dst_dir)
    return dst_dir

if CODE_INPUT_DIR and not (PROJECT_DIR / 'src' / 'main.py').exists():
    PROJECT_DIR = copy_project_from_input(CODE_INPUT_DIR, PROJECT_DIR)

def find_project_code_in_kaggle_input():
    if not KAGGLE_INPUT.exists():
        return None

    patterns = [
        '*/src/main.py',
        '*/*/src/main.py',
        '*/*/*/src/main.py',
    ]
    for pattern in patterns:
        for main_file in KAGGLE_INPUT.glob(pattern):
            return main_file.parent.parent

    for main_file in KAGGLE_INPUT.rglob('src/main.py'):
        return main_file.parent.parent

    return None

if not (PROJECT_DIR / 'src' / 'main.py').exists():
    found_project = find_project_code_in_kaggle_input()
    if found_project is not None:
        PROJECT_DIR = copy_project_from_input(found_project, PROJECT_DIR)

if not (PROJECT_DIR / 'src' / 'main.py').exists() and GITHUB_REPO_URL:
    subprocess.check_call(['git', 'clone', '--depth', '1', GITHUB_REPO_URL, str(PROJECT_DIR)])

if not (PROJECT_DIR / 'src' / 'main.py').exists():
    raise FileNotFoundError(
        'Could not find project code. Attach your repository as a Kaggle input dataset, '
        'upload/copy it to /kaggle/working/Classification, set CODE_INPUT_DIR to the '
        'folder containing src/main.py, or set GITHUB_REPO_URL to clone it.'
    )

sys.path.insert(0, str(PROJECT_DIR))
print('Project:', PROJECT_DIR)

In [ ]:
CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

# Optional manual setting if auto-detection cannot pick the right Kaggle input.
# Example: ISIC18_INPUT_DIR = '/kaggle/input/your-dataset-name'
ISIC18_INPUT_DIR = os.environ.get('ISIC18_INPUT_DIR', '').strip()

def is_valid_class_dir(parent):
    parent = Path(parent)
    return all((parent / class_name).is_dir() for class_name in CLASS_NAMES)

def validate_isic18_root(path):
    path = Path(path)
    split_dirs = {
        'train': path / 'ham10000_80_20' / 'ham10000_80_20' / 'train_dir',
        'val': path / 'Valid' / 'kaggle' / 'working' / 'Valid',
        'test': path / 'Test' / 'kaggle' / 'working' / 'Test',
    }
    missing = []
    for split_name, split_dir in split_dirs.items():
        if not split_dir.exists():
            missing.append(str(split_dir))
            continue
        for class_name in CLASS_NAMES:
            class_dir = split_dir / class_name
            if not class_dir.exists():
                missing.append(str(class_dir))
    if missing:
        preview = '\n'.join(missing[:20])
        raise FileNotFoundError(f'ISIC18 dataset structure mismatch under {path}. Missing:\n{preview}')
    return split_dirs

def find_isic18_root():
    if ISIC18_INPUT_DIR:
        path = Path(ISIC18_INPUT_DIR)
        validate_isic18_root(path)
        return path

    candidates = []
    if KAGGLE_INPUT.exists():
        candidates.extend(path for path in KAGGLE_INPUT.iterdir() if path.is_dir())

    for path in candidates:
        train_dir = path / 'ham10000_80_20' / 'ham10000_80_20' / 'train_dir'
        val_dir = path / 'Valid' / 'kaggle' / 'working' / 'Valid'
        test_dir = path / 'Test' / 'kaggle' / 'working' / 'Test'
        if is_valid_class_dir(train_dir) and is_valid_class_dir(val_dir) and is_valid_class_dir(test_dir):
            return path

    raise FileNotFoundError(
        'Could not find ISIC18 dataset under /kaggle/input. Expected a Kaggle input root containing '
        'ham10000_80_20/ham10000_80_20/train_dir, Valid/kaggle/working/Valid, '
        'and Test/kaggle/working/Test. Set ISIC18_INPUT_DIR manually if needed.'
    )

ISIC18_SOURCE_DIR = find_isic18_root()
split_dirs = validate_isic18_root(ISIC18_SOURCE_DIR)

# src/data/dataset_config.py expects CLASSIFICATION_DATA_ROOT/isic18.
# Build a lightweight normalized view with Train/Valid/Test links.
DATA_ROOT = KAGGLE_WORKING / 'data'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
ISIC18_LINK = DATA_ROOT / 'isic18'
ISIC18_LINK.mkdir(parents=True, exist_ok=True)

def relink(dst, src):
    dst = Path(dst)
    src = Path(src)
    if dst.is_symlink() or dst.is_file():
        dst.unlink()
    elif dst.exists():
        shutil.rmtree(dst)
    dst.symlink_to(src, target_is_directory=True)

relink(ISIC18_LINK / 'Train', split_dirs['train'])
relink(ISIC18_LINK / 'Valid', split_dirs['val'])
relink(ISIC18_LINK / 'Test', split_dirs['test'])

os.environ['CLASSIFICATION_DATA_ROOT'] = str(DATA_ROOT)

print('ISIC18 source:', ISIC18_SOURCE_DIR)
print('Train dir:', split_dirs['train'])
print('Val dir:', split_dirs['val'])
print('Test dir:', split_dirs['test'])
print('ISIC18 normalized root:', ISIC18_LINK)
print('CLASSIFICATION_DATA_ROOT:', os.environ['CLASSIFICATION_DATA_ROOT'])


In [ ]:
# Main experiment controls. Defaults are for ISIC18 + EDF learnable fusion.
DATASET = 'isic18'
MODEL = 'edf'
BACKBONE = 'resnet101'
EXPERT_MODE = 'multi_layer'
DISAGREEMENT_TYPE = 'learnable'

EPOCHS = 100
FREEZE_EPOCHS = 5
BATCH_SIZE = 16
LR = 1e-4
LOSS = 'focal'
SCHEDULER = 'cosine'
IMG_SIZE = 224
PROJ_DIM = 224
NUM_WORKERS = 2
SEED = 42
USE_AMP = True

OUTPUT_DIR = '/kaggle/working/outputs'

In [ ]:
# CUDA preflight. If Kaggle assigns a newer GPU and Torch is too old,
# conv2d can fail with: no kernel image is available for execution on the device.
INSTALL_TORCH_CU128 = False

if INSTALL_TORCH_CU128:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--upgrade',
        'torch', 'torchvision', 'torchaudio',
        '--index-url', 'https://download.pytorch.org/whl/cu128',
    ])

import torch

print('Torch:', torch.__version__)
print('Torch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Capability:', torch.cuda.get_device_capability(0))


In [ ]:
cmd = [
    sys.executable, str(PROJECT_DIR / 'src' / 'main.py'),
    '--output-dir', OUTPUT_DIR,
    '--dataset', DATASET,
    '--model', MODEL,
    '--scheduler', SCHEDULER,
    '--backbone1', BACKBONE,
    '--expert-mode', EXPERT_MODE,
    '--disagreement-type', DISAGREEMENT_TYPE,
    '--proj-dim', str(PROJ_DIM),
    '--epochs', str(EPOCHS),
    '--loss', LOSS,
    '--batch-size', str(BATCH_SIZE),
    '--lr', str(LR),
    '--img-size', str(IMG_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--seed', str(SEED),
    '--freeze-epochs', str(FREEZE_EPOCHS),
]

if USE_AMP:
    cmd.append('--amp')

print('Running:')
print(' '.join(cmd))

env = os.environ.copy()
subprocess.run(cmd, cwd=PROJECT_DIR, env=env, check=True)

In [ ]:
results_dir = Path(OUTPUT_DIR) / 'results' / DATASET
print('Results directory:', results_dir)
for path in sorted(results_dir.glob('*')):
    print(path)